# Avance 3. Baseline MLOps para sistema de recomendaciones de pricing

Este notebook construye una línea base operativa para monitorear un sistema de recomendaciones de precios B2B.

A diferencia de un baseline supervisado tradicional, este avance no busca entrenar un modelo simple para reemplazar el motor de pricing.

Por lo tanto, el baseline se define como una fotografía inicial del comportamiento esperado del sistema. Esta línea base permite comparar futuras corridas, detectar drift, evaluar calidad de datos, monitorear cambios en catálogos y activar un semáforo operativo Green/Yellow/Red.

## 1. Carga de librerías y configuración

El notebook parte de los artefactos generados en el Avance 2:

- `feature_table`: tabla principal para el baseline MLOps.
- `order_size_catalog`: catálogo de tamaño de orden.
- `disty_margin_catalog`: catálogo de margen del distribuidor.
- `channel_margin_catalog`: catálogo de margen del canal.
- `catalog_metadata`: metadatos de versionado de los catálogos.

La ruta se configuró para funcionar tanto en Google Colab como en un entorno local.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 120)

BASELINE_VERSION = "baseline_mlops_pricing_v1"
RUN_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")

# Ruta en Google Colab. Ajustar si los archivos están en otra carpeta.
COLAB_DIR_PATH = "/content/drive/MyDrive/Maestria/Yageo-MLOPS"

# Ruta local usada cuando se ejecuta fuera de Colab.
LOCAL_DIR_PATH = "/mnt/data"

if os.path.exists(COLAB_DIR_PATH):
    DIR_PATH = COLAB_DIR_PATH
else:
    DIR_PATH = LOCAL_DIR_PATH

print("Using DIR_PATH:", DIR_PATH)

Using DIR_PATH: /content/drive/MyDrive/Maestria/Yageo-MLOPS


In [5]:
FEATURE_TABLE_PATH = f"{DIR_PATH}/avance2_feature_table_equipo46_v6_mlops_sanitized.csv"
ORDER_SIZE_CATALOG_PATH = f"{DIR_PATH}/avance2_order_size_catalog_equipo46_v6_mlops_sanitized.csv"
DISTY_MARGIN_CATALOG_PATH = f"{DIR_PATH}/avance2_disty_margin_catalog_equipo46_v6_mlops_sanitized.csv"
CHANNEL_MARGIN_CATALOG_PATH = f"{DIR_PATH}/avance2_channel_margin_catalog_equipo46_v6_mlops_sanitized.csv"
CATALOG_METADATA_PATH = f"{DIR_PATH}/avance2_catalog_metadata_equipo46_v6_mlops_sanitized.csv"

feature_table = pd.read_csv(FEATURE_TABLE_PATH, low_memory=False)
order_size_catalog = pd.read_csv(ORDER_SIZE_CATALOG_PATH, low_memory=False)
disty_margin_catalog = pd.read_csv(DISTY_MARGIN_CATALOG_PATH, low_memory=False)
channel_margin_catalog = pd.read_csv(CHANNEL_MARGIN_CATALOG_PATH, low_memory=False)
catalog_metadata = pd.read_csv(CATALOG_METADATA_PATH, low_memory=False)

print("Feature table:", feature_table.shape)
print("Order size catalog:", order_size_catalog.shape)
print("Disty margin catalog:", disty_margin_catalog.shape)
print("Channel margin catalog:", channel_margin_catalog.shape)
print("Catalog metadata:", catalog_metadata.shape)

display(feature_table.head())
display(catalog_metadata)

Feature table: (8314, 104)
Order size catalog: (64493, 10)
Disty margin catalog: (634, 9)
Channel margin catalog: (27, 8)
Catalog metadata: (3, 5)


,kpn,vpareadescription,distysegment,mean_elasticity,P0_PRICE,P20_PRICE,P50_PRICE,P85_PRICE,P100_PRICE,P0_RESALE,P20_RESALE,P50_RESALE,P85_RESALE,P100_RESALE,P0_QTY,P20_QTY,P35_QTY,P50_QTY,P65_QTY,P85_QTY,P100_QTY,P20,Min_P20_for_5pct_margin,P20_Adjusted_Min5pctMargin,P20_Was_Adjusted,P50,P85,q0,q0_method,q0_local_obs,Revenue_Aggressive,Revenue_Aggressive_QTY_OPT,More_Revenue,More_Revenue_QTY_OPT,Balanced,Balanced_QTY_OPT,More_Profit,More_Profit_QTY_OPT,Profit_Aggressive,Profit_Aggressive_QTY_OPT,s_curve_price_range_p85_p20,s_curve_center_gap_p50_p20,s_curve_upper_gap_p85_p50,s_curve_range_ratio_p85_p20,recommended_below_p20,recommended_above_p85,recommended_outside_p20_p85,recommended_position_p20_p85,recommended_gap_vs_p50,recommended_gap_vs_p50_pct,recommended_near_extreme_zone,recommended_historical_zone,strategy_price_min,strategy_price_max,strategy_price_range,strategy_price_range_ratio,strategy_qty_min,strategy_qty_max,strategy_qty_range,p20_adjustment_amount,p20_adjustment_pct,price_percentiles_monotonic,qty_percentiles_monotonic,q0_local_obs_bin,q0_low_support_flag,n_transactions,n_invoices,revenue_sum,revenue_median,resale_price_median,resale_price_mean,into_stock_price_median,quantity_median,quantity_sum,log_quantity_mean,log_into_stock_price_mean,distributor_margin_pct_median,channel_margin_share_median,kemet_margin_pct_median,negative_distributor_margin_rate,negative_kemet_variable_margin_rate,channel_margin_share_out_of_range_rate,auth_cost_match_rate,elasticity_mean_avg,elasticity_sd_avg,elasticity_hdi_width_avg,order_size_bin_mode,disty_margin_original_bin_mode,channel_margin_share_bin_mode,date_min,date_max,n_distributor_parents,distributor_parentnumber_mode,distributor_parent_top_share,custombusinessgroup_mode,custombusinessgroup_top_share,log1p_revenue_sum,log1p_quantity_sum,log1p_n_transactions,kpn_frequency,vpareadescription_frequency,distysegment_frequency,custombusinessgroup_mode_frequency,distributor_parentnumber_mode_frequency
0,KPN_00000001,VPAREA_00000002,DISTSEG_00000002,-3.764863,9.848737,9.848737,10.362581,10.404101,10.410032,12.618120,12.801119,13.107570,13.388520,13.784039,6000.0,12000.0,12000.0,12000.0,12000.0,12000.0,12000.0,9.848737,6.168000,9.848737,False,10.362581,10.404101,11000.0,local_avg,6,9.848737,13030.020995,9.848737,13030.020995,9.848737,13030.020995,9.848737,13030.020995,9.848737,13030.020995,0.555364,0.513844,0.041520,1.056389,0,0,0,0.0,-0.513844,-0.049586,1,Near_P20,9.848737,9.848737,0.0,1.0,13030.020995,13030.020995,0.0,0.0,0.0,1,1,Medium,0,8,8,1.183797e+06,157345.209000,13.112101,13.131285,10.362581,12000.0,90000.0,9.306019,2.325722,0.214858,0.613404,0.434542,0.0,0.0,0.0,1.0,-3.764863,0.705578,2.625786,Missing,Missing,Medium,2023-08-28,2025-05-27,1,DISTNUM_00000002,1.0,BUSINESSGRP_00000003,1.0,13.984238,11.407576,2.197225,0.000034,0.610649,0.994572,0.020887,0.162336
1,KPN_00000002,VPAREA_00000002,DISTSEG_00000002,-1.180541,7.981791,7.981791,7.981791,7.981791,7.981791,9.444840,9.444840,10.038600,10.632360,10.632360,5832.0,5832.0,5832.0,5832.0,5832.0,5832.0,5832.0,7.981791,4.304614,7.981791,False,7.981791,7.981791,5832.0,local_avg,5,7.981791,5832.000000,7.981791,5832.000000,7.981791,5832.000000,7.981791,5832.000000,7.981791,5832.000000,0.000000,0.000000,0.000000,1.000000,0,0,0,NaN,0.000000,0.000000,0,Missing,7.981791,7.981791,0.0,1.0,5832.000000,5832.000000,0.0,0.0,0.0,1,1,Low,0,5,5,2.961884e+05,62007.923520,10.632360,10.157352,7.981791,5832.0,29160.0,8.671115,2.077163,0.249293,0.594899,0.487661,0.0,0.0,0.0,1.0,-1.180541,1.253652,4.721376,Missing,Missing,Low,2025-03-12,2025-07-21,1,DISTNUM_00000002,1.0,BUSINESSGRP_00000002,1.0,12.598754,10.280587,1.791759,0.000021,0.610649,0.994572,0.083532,0.162336
2,KPN_00000003,VPAREA_00000002,DISTSEG_00000002,-1.688701,10.775419,10.775419,10.775419,10.775419,10.775419,12.743041,12.743041,13.083816,13.967561,14.345281,5160.0,5160.0,5160.0,5160.0,5160.0,9133.2,12384.0,10.775419,6.701495,10.775419,False,10.775419,10.775419,6966.0,local_avg,4,10.7

,artifact,source,version,rows,keys
0,order_size_catalog,general_transactions_masked,catalog_general_masked_v1,64493,kpn | vpareadescription | distysegment
1,disty_margin_catalog,general_transactions_masked,catalog_general_masked_v1,634,hier_lev1 | vpareadescription | custombusiness...
2,channel_margin_catalog,general_transactions_masked,catalog_general_masked_v1,27,custombusinessgroup | distysegment | vpareades...


## 2. Validación de columnas críticas

Esta sección verifica que la tabla generada en el Avance 2 contiene las columnas necesarias para construir el baseline MLOps.

Las columnas críticas incluyen llaves de granularidad, elasticidad, percentiles históricos, señales del output, flags de calidad, bins y métricas de impacto económico.

In [6]:
critical_columns = [
    "kpn",
    "vpareadescription",
    "distysegment",
    "mean_elasticity",
    "P20_PRICE",
    "P50_PRICE",
    "P85_PRICE",
    "P20_Was_Adjusted",
    "recommended_outside_p20_p85",
    "recommended_near_extreme_zone",
    "recommended_position_p20_p85",
    "recommended_historical_zone",
    "s_curve_price_range_p85_p20",
    "price_percentiles_monotonic",
    "qty_percentiles_monotonic",
    "q0_low_support_flag",
    "q0_local_obs",
    "order_size_bin_mode",
    "disty_margin_original_bin_mode",
    "channel_margin_share_bin_mode",
    "revenue_sum",
    "n_transactions"
]

missing_critical_columns = [col for col in critical_columns if col not in feature_table.columns]

print("Columnas críticas faltantes:")
print(missing_critical_columns)

print("\nColumnas críticas disponibles:")
print([col for col in critical_columns if col in feature_table.columns])

assert len(missing_critical_columns) == 0, "Faltan columnas críticas para el baseline."

Columnas críticas faltantes:
[]

Columnas críticas disponibles:
['kpn', 'vpareadescription', 'distysegment', 'mean_elasticity', 'P20_PRICE', 'P50_PRICE', 'P85_PRICE', 'P20_Was_Adjusted', 'recommended_outside_p20_p85', 'recommended_near_extreme_zone', 'recommended_position_p20_p85', 'recommended_historical_zone', 's_curve_price_range_p85_p20', 'price_percentiles_monotonic', 'qty_percentiles_monotonic', 'q0_low_support_flag', 'q0_local_obs', 'order_size_bin_mode', 'disty_margin_original_bin_mode', 'channel_margin_share_bin_mode', 'revenue_sum', 'n_transactions']


## 3. Métricas clave del baseline

Estas métricas resumen la corrida base del sistema. Funcionan como punto de referencia para comparar futuras corridas.

En esta primera versión, se mide:

- Número de registros y granularidad.
- Proporción de ajustes de P20.
- Proporción de recomendaciones cerca de extremos.
- Consistencia de percentiles.
- Soporte histórico bajo.
- Proporción de posiciones no calculables dentro de P20-P85.

In [7]:
baseline_key_metrics = pd.DataFrame([
    {"metric": "n_rows", "value": len(feature_table)},
    {"metric": "n_unique_kpn", "value": feature_table["kpn"].nunique()},
    {"metric": "n_unique_vparea", "value": feature_table["vpareadescription"].nunique()},
    {"metric": "n_unique_distysegment", "value": feature_table["distysegment"].nunique()},
    {"metric": "p20_was_adjusted_rate", "value": feature_table["P20_Was_Adjusted"].mean()},
    {"metric": "recommended_outside_p20_p85_original_rate", "value": feature_table["recommended_outside_p20_p85"].mean()},
    {"metric": "recommended_near_extreme_zone_rate", "value": feature_table["recommended_near_extreme_zone"].mean()},
    {"metric": "price_percentiles_monotonic_rate", "value": feature_table["price_percentiles_monotonic"].mean()},
    {"metric": "qty_percentiles_monotonic_rate", "value": feature_table["qty_percentiles_monotonic"].mean()},
    {"metric": "q0_low_support_rate", "value": feature_table["q0_low_support_flag"].mean()},
    {"metric": "recommended_position_missing_rate", "value": feature_table["recommended_position_p20_p85"].isna().mean()}
])

baseline_key_metrics

,metric,value
0,n_rows,8314.000000
1,n_unique_kpn,6982.000000
2,n_unique_vparea,2.000000
3,n_unique_distysegment,2.000000
4,p20_was_adjusted_rate,0.003127
5,recommended_outside_p20_p85_original_rate,0.000842
6,recommended_near_extreme_zone_rate,0.381285
7,price_percentiles_monotonic_rate,1.000000
8,qty_percentiles_monotonic_rate,1.000000
9,q0_low_support_rate,0.464638


### Interpretación inicial

La tabla final tiene una estructura consistente para monitoreo MLOps. Las columnas críticas están disponibles y las validaciones de monotonicidad de percentiles de precio y cantidad permiten evaluar la consistencia interna de las bandas históricas.

La variable `recommended_position_p20_p85` puede tener valores faltantes cuando la banda histórica P20-P85 no es informativa, por ejemplo cuando P20 y P85 son iguales o prácticamente iguales. Esto no implica necesariamente un error del sistema; representa baja variabilidad histórica de precio o segmentos donde la transacción se concentra en un solo nivel de precio.

## 4. Baseline de calidad de datos

Esta sección mide nulos, cardinalidad y tipos de datos para las variables principales del baseline.

El objetivo no es solamente encontrar errores, sino identificar señales de cobertura y monitoreo, por ejemplo:

- Campos con alta proporción de `Missing`.
- Bins no asignados.
- Variables que deben tratarse como `monitor_only`.

In [8]:
monitoring_features = [
    "mean_elasticity",
    "P20_PRICE",
    "P50_PRICE",
    "P85_PRICE",
    "P20_Was_Adjusted",
    "recommended_outside_p20_p85",
    "recommended_near_extreme_zone",
    "recommended_position_p20_p85",
    "recommended_historical_zone",
    "s_curve_price_range_p85_p20",
    "price_percentiles_monotonic",
    "qty_percentiles_monotonic",
    "q0_low_support_flag",
    "q0_local_obs",
    "n_transactions",
    "revenue_sum",
    "order_size_bin_mode",
    "disty_margin_original_bin_mode",
    "channel_margin_share_bin_mode",
    "distributor_margin_pct_median",
    "channel_margin_share_median",
    "kemet_margin_pct_median",
    "negative_distributor_margin_rate",
    "negative_kemet_variable_margin_rate",
    "channel_margin_share_out_of_range_rate",
    "auth_cost_match_rate"
]

monitoring_features = [col for col in monitoring_features if col in feature_table.columns]

quality_summary = pd.DataFrame({
    "feature": monitoring_features,
    "dtype": [feature_table[col].dtype for col in monitoring_features],
    "missing_count": [feature_table[col].isna().sum() for col in monitoring_features],
    "missing_pct": [feature_table[col].isna().mean() for col in monitoring_features],
    "unique_count": [feature_table[col].nunique(dropna=True) for col in monitoring_features]
})

quality_summary.sort_values("missing_pct", ascending=False)

,feature,dtype,missing_count,missing_pct,unique_count
7,recommended_position_p20_p85,float64,4789,0.576016,423
0,mean_elasticity,float64,0,0.000000,6981
1,P20_PRICE,float64,0,0.000000,5841
2,P50_PRICE,float64,0,0.000000,5924
4,P20_Was_Adjusted,bool,0,0.000000,2
3,P85_PRICE,float64,0,0.000000,5953
5,recommended_outside_p20_p85,int64,0,0.000000,2
6,recommended_near_extreme_zone,int64,0,0.000000,2
8,recommended_historical_zone,object,0,0.000000,4
9,s_curve_price_range_p85_p20,float64,0,0.000000,3360


## 5. Baseline numérico

Esta sección resume la distribución base de las variables numéricas principales. Estas distribuciones sirven como referencia para drift futuro.

Variables como precio, elasticidad, revenue, número de transacciones y márgenes no se usan aquí para entrenar un modelo nuevo, sino para establecer una línea base de estabilidad operativa.

In [9]:
numeric_features = [
    "mean_elasticity",
    "P20_PRICE",
    "P50_PRICE",
    "P85_PRICE",
    "recommended_position_p20_p85",
    "s_curve_price_range_p85_p20",
    "q0_local_obs",
    "n_transactions",
    "revenue_sum",
    "distributor_margin_pct_median",
    "channel_margin_share_median",
    "kemet_margin_pct_median",
    "auth_cost_match_rate"
]

numeric_features = [col for col in numeric_features if col in feature_table.columns]

numeric_baseline = feature_table[numeric_features].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

numeric_baseline

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
mean_elasticity,8314.0,-2.358694e+00,2.662961e+00,-17.528416,-10.035492,-6.867792,-3.796072,-2.177723,-8.478163e-01,1.498548e+00,4.400329e+00,1.395444e+01
P20_PRICE,8314.0,9.259554e+00,4.348133e+01,0.014400,0.022800,0.032943,0.143995,0.867008,4.306200e+00,3.544126e+01,1.606697e+02,1.369119e+03
P50_PRICE,8314.0,9.366352e+00,4.369126e+01,0.014400,0.022802,0.033600,0.150000,0.887400,4.389984e+00,3.544126e+01,1.611077e+02,1.369119e+03
P85_PRICE,8314.0,9.536476e+00,4.422235e+01,0.014400,0.024000,0.036000,0.162000,0.927480,4.529700e+00,3.600000e+01,1.619064e+02,1.369119e+03
recommended_position_p20_p85,3525.0,2.228250e-01,3.896719e-01,0.000000,0.000000,0.000000,0.000000,0.000000,2.802803e-01,1.000000e+00,1.000000e+00,1.000000e+00
s_curve_price_range_p85_p20,8314.0,2.769222e-01,2.216637e+00,0.000000,0.000000,0.000000,0.000000,0.000000,2.400000e-02,8.184467e-01,5.194530e+00,6.787123e+01
q0_local_obs,8314.0,1.742483e+01,8.001579e+01,1.000000,1.000000,1.000000,2.000000,5.000000,1.600000e+01,6.200000e+01,1.428700e+02,4.626000e+03
n_transactions,8314.0,2.818788e+01,1.097390e+02,1.000000,1.000000,1.000000,2.000000,7.000000,2.300000e+01,1.060000e+02,2.974800e+02,5.138000e+03
revenue_sum,8314.0,3.465366e+06,1.591650e+07,1687.679904,7256.304000,23616.000000,150297.118200,620286.350400,2.317672e+06,1.331432e+07,4.517625e+07,7.016925e+08
distributor_margin_pct_median,8314.0,1.749213e-01,9.260662e-02,0.001572,0.029412,0.039299,0.108135,0.175178,2.285045e-01,3.133968e-01,4.641482e-01,9.367926e-01


## 6. Baseline categórico y binario

Las variables categóricas y binarias permiten monitorear la composición del sistema: zonas históricas, soporte bajo, bins de tamaño de orden, bins de margen y flags del output.

Estas distribuciones también serán útiles para calcular PSI categórico en futuras corridas.

In [10]:
categorical_features = [
    "recommended_historical_zone",
    "q0_low_support_flag",
    "order_size_bin_mode",
    "disty_margin_original_bin_mode",
    "channel_margin_share_bin_mode",
    "P20_Was_Adjusted",
    "recommended_outside_p20_p85",
    "recommended_near_extreme_zone",
    "price_percentiles_monotonic",
    "qty_percentiles_monotonic"
]

categorical_features = [col for col in categorical_features if col in feature_table.columns]

categorical_baseline_tables = {}

for col in categorical_features:
    table = (
        feature_table[col]
        .astype("object")
        .fillna("Missing")
        .value_counts(normalize=True)
        .rename("proportion")
        .to_frame()
    )
    categorical_baseline_tables[col] = table
    print(f"\n--- {col} ---")
    display(table)


--- recommended_historical_zone ---


,proportion
recommended_historical_zone,
Missing,0.576016
Near_P20,0.296969
Mid_to_P85,0.091532
P20_to_Mid,0.035482



--- q0_low_support_flag ---


/tmp/ipykernel_3431/648873952.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("Missing")


,proportion
q0_low_support_flag,
0,0.535362
1,0.464638


...

/tmp/ipykernel_3431/648873952.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("Missing")


,proportion
price_percentiles_monotonic,
1,1.0



--- qty_percentiles_monotonic ---


/tmp/ipykernel_3431/648873952.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna("Missing")


,proportion
qty_percentiles_monotonic,
1,1.0


## 7. Corrección conceptual: banda P20-P85 válida vs banda colapsada

Durante el análisis se identificó que no conviene interpretar directamente `recommended_outside_p20_p85` como una señal crítica, porque esta variable puede mezclar dos situaciones distintas:

1. Una recomendación realmente fuera de una banda histórica válida.
2. Una banda P20-P85 colapsada, donde P20 y P85 son iguales o prácticamente iguales.

Cuando P20 y P85 son iguales, no existe una banda histórica útil para evaluar si la recomendación está dentro o fuera del rango. En esos casos, el problema no es necesariamente una recomendación incorrecta, sino una limitación de cobertura o variabilidad histórica.

Por ello, se crean nuevas etiquetas:

- `price_band_is_valid`: existe una banda P20-P85 útil.
- `price_band_is_collapsed`: P20 y P85 son iguales o prácticamente iguales.
- `recommended_outside_valid_p20_p85`: outside real, solo evaluado cuando la banda es válida.
- `recommended_band_not_evaluable`: la banda está colapsada y no se puede evaluar outside de forma confiable.

In [11]:
EPS = 1e-9

# La columna "Balanced" representa una de las recomendaciones finales generadas por el motor.
# Si la organización define otra recomendación final oficial, ajustar esta variable.
recommended_price_col = "Balanced"

assert recommended_price_col in feature_table.columns, f"No existe la columna {recommended_price_col}"

feature_table["price_band_width"] = feature_table["P85_PRICE"] - feature_table["P20_PRICE"]

feature_table["price_band_is_collapsed"] = (
    feature_table["price_band_width"].abs() <= EPS
).astype(int)

feature_table["price_band_is_valid"] = (
    feature_table["price_band_width"].abs() > EPS
).astype(int)

feature_table["recommended_outside_valid_p20_p85"] = (
    (feature_table["price_band_is_valid"] == 1) &
    (
        (feature_table[recommended_price_col] < feature_table["P20_PRICE"]) |
        (feature_table[recommended_price_col] > feature_table["P85_PRICE"])
    )
).astype(int)

feature_table["recommended_band_not_evaluable"] = (
    feature_table["price_band_is_collapsed"] == 1
).astype(int)

band_validation_summary = feature_table[[
    "price_band_is_valid",
    "price_band_is_collapsed",
    "recommended_outside_p20_p85",
    "recommended_outside_valid_p20_p85",
    "recommended_band_not_evaluable"
]].mean().rename("proportion").to_frame()

band_validation_summary

,proportion
price_band_is_valid,0.423984
price_band_is_collapsed,0.576016
recommended_outside_p20_p85,0.000842
recommended_outside_valid_p20_p85,0.000000
recommended_band_not_evaluable,0.576016


### Interpretación

La métrica `price_band_is_collapsed` se mantiene como indicador `monitor_only`, no como error. Una banda colapsada puede significar que el producto-segmento se vende históricamente a un solo precio o tiene poca variabilidad de precio.

El estado Red se reserva para un caso más fuerte: una recomendación fuera de una banda P20-P85 válida.

## 8. Peso económico por revenue

No todos los registros deben tener el mismo peso operativo. Un caso con alto impacto económico debe revisarse antes que un caso de bajo impacto, incluso si ambos tienen la misma severidad técnica.

Por ello, se crea un bucket de impacto económico usando `revenue_sum`:

- `Low_Impact`: hasta percentil 80.
- `Medium_Impact`: percentil 80 a 95.
- `High_Impact`: percentil 95 a 99.
- `Critical_Impact`: percentil 99 a 100.

In [12]:
feature_table["revenue_percentile"] = feature_table["revenue_sum"].rank(
    pct=True,
    ascending=True
)

feature_table["revenue_weight_bucket"] = pd.cut(
    feature_table["revenue_percentile"],
    bins=[0, 0.80, 0.95, 0.99, 1.00],
    labels=["Low_Impact", "Medium_Impact", "High_Impact", "Critical_Impact"],
    include_lowest=True
)

feature_table["revenue_weight_bucket"].value_counts(normalize=True).rename("proportion").to_frame()

,proportion
revenue_weight_bucket,
Low_Impact,0.799976
Medium_Impact,0.149988
High_Impact,0.039933
Critical_Impact,0.010103


## 9. Semáforo técnico final y priority score

El semáforo técnico evalúa la severidad de la recomendación:

- **Red**: recomendación fuera de una banda P20-P85 válida.
- **Yellow**: recomendación cerca de extremos o P20 ajustado.
- **Green**: sin alerta directa.

Las señales de cobertura, como banda colapsada o bajo soporte histórico, no cambian automáticamente el semáforo a Red. Se incorporan al score de prioridad para ordenar la revisión.

El priority score combina:

- Severidad técnica.
- Señales de cobertura/soporte.
- Impacto económico por revenue.

In [13]:
def classify_recommendation_status_final(row):
    if row["recommended_outside_valid_p20_p85"] == 1:
        return "Red"

    if row["recommended_near_extreme_zone"] == 1:
        return "Yellow"

    if row["P20_Was_Adjusted"] == True:
        return "Yellow"

    return "Green"


def add_priority_score_final(df):
    scored = df.copy()
    scored["priority_score"] = 0

    # Riesgo real: fuera de banda válida
    scored["priority_score"] += scored["recommended_outside_valid_p20_p85"].astype(int) * 5

    # Señales de observación del output
    scored["priority_score"] += scored["recommended_near_extreme_zone"].astype(int) * 3
    scored["priority_score"] += scored["P20_Was_Adjusted"].astype(int) * 2

    # Señales de cobertura/soporte
    scored["priority_score"] += scored["price_band_is_collapsed"].astype(int) * 1
    scored["priority_score"] += scored["q0_low_support_flag"].astype(int) * 1

    # Peso económico
    impact_points = {
        "Low_Impact": 0,
        "Medium_Impact": 1,
        "High_Impact": 2,
        "Critical_Impact": 3
    }

    scored["impact_points"] = (
        scored["revenue_weight_bucket"]
        .astype(str)
        .map(impact_points)
        .fillna(0)
    )

    scored["priority_score"] += scored["impact_points"]

    return scored


def classify_review_priority_final(score):
    if score >= 8:
        return "High"
    elif score >= 5:
        return "Medium"
    else:
        return "Low"


feature_table_scored_final = add_priority_score_final(feature_table)

feature_table_scored_final["recommendation_status_final"] = feature_table_scored_final.apply(
    classify_recommendation_status_final,
    axis=1
)

feature_table_scored_final["review_priority_final"] = (
    feature_table_scored_final["priority_score"]
    .apply(classify_review_priority_final)
)

display(feature_table_scored_final["recommendation_status_final"].value_counts(normalize=True).rename("proportion").to_frame())
display(feature_table_scored_final["review_priority_final"].value_counts(normalize=True).rename("proportion").to_frame())
display(feature_table_scored_final["priority_score"].describe().to_frame().T)

,proportion
recommendation_status_final,
Green,0.61631
Yellow,0.38369


,proportion
review_priority_final,
Low,0.954294
Medium,0.045706


,count,mean,std,min,25%,50%,75%,max
priority_score,8314.0,2.450926,1.192042,0.0,2.0,2.0,3.0,7.0


### Interpretación del semáforo final

Después de separar los casos con banda P20-P85 válida de los casos con banda colapsada, el semáforo final permite distinguir entre riesgo real del output y limitaciones de información histórica.

Si el motor de recomendación está diseñado para operar dentro de P20-P85, entonces no se espera que existan recomendaciones fuera de una banda válida bajo condiciones normales. La aparición de casos Red en futuras corridas indicaría una de tres posibilidades:

1. Cambio en la lógica de optimización.
2. Ajustes posteriores como margen mínimo, floor, price point o reglas comerciales.
3. Problemas en el cálculo de percentiles o rangos históricos.

Los casos con `price_band_is_collapsed = 1` se tratan como señal de cobertura o baja variabilidad histórica, no como error crítico.

## 10. Cola de revisión priorizada

Esta tabla ordena los casos que conviene revisar primero. No reemplaza el juicio del equipo de pricing, pero reduce la carga operativa al priorizar por severidad técnica e impacto económico.

In [14]:
review_queue = feature_table_scored_final.sort_values(
    ["priority_score", "revenue_sum"],
    ascending=[False, False]
)

review_queue_cols = [
    "kpn",
    "vpareadescription",
    "distysegment",
    "revenue_sum",
    "revenue_weight_bucket",
    "recommendation_status_final",
    "review_priority_final",
    "priority_score",
    "recommended_outside_valid_p20_p85",
    "price_band_is_collapsed",
    "recommended_near_extreme_zone",
    "P20_Was_Adjusted",
    "q0_low_support_flag",
    "recommended_historical_zone"
]

review_queue[review_queue_cols].head(30)

,kpn,vpareadescription,distysegment,revenue_sum,revenue_weight_bucket,recommendation_status_final,review_priority_final,priority_score,recommended_outside_valid_p20_p85,price_band_is_collapsed,recommended_near_extreme_zone,P20_Was_Adjusted,q0_low_support_flag,recommended_historical_zone
4944,KPN_00004113,VPAREA_00000002,DISTSEG_00000002,6.245671e+07,Critical_Impact,Yellow,Medium,7,0,0,1,False,1,P20_to_Mid
7967,KPN_00006680,VPAREA_00000002,DISTSEG_00000002,5.892624e+07,Critical_Impact,Yellow,Medium,7,0,0,1,False,1,Near_P20
7966,KPN_00006680,VPAREA_00000001,DISTSEG_00000002,5.874409e+07,Critical_Impact,Yellow,Medium,7,0,0,1,False,1,Near_P20
2741,KPN_00002287,VPAREA_00000001,DISTSEG_00000002,2.169720e+07,High_Impact,Yellow,Medium,7,0,0,1,True,0,P20_to_Mid
2974,KPN_00002472,VPAREA_00000002,DISTSEG_00000002,1.280786e+07,Medium_Impact,Yellow,Medium,7,0,0,1,True,1,P20_to_Mid
5475,KPN_00004544,VPAREA_00000002,DISTSEG_00000002,5.843981e+08,Critical_Impact,Yellow,Medium,6,0,0,1,False,0,Near_P20
4718,KPN_00003913,VPAREA_00000002,DISTSEG_00000002,5.004838e+08,Critical_Impact,Yellow,Medium,6,0,0,1,False,0,Near_P20
7978,KPN_00006688,VPAREA_00000001,DISTSEG_00000002,3.545833e+08,Critical_Impact,Yellow,Medium,6,0,0,1,False,0,Near_P20
888,KPN_00000794,VPAREA_00000002,DISTSEG_00000002,3.232394e+08,Critical_Impact,Yellow,Medium,6,0,0,1,False,0,Near_P20
7909,KPN_00006635,VPAREA_00000002,DISTSEG_00000002,2.967712e+08,Critical_Impact,Yellow,Medium,6,0,0,1,False,0,P20_to_Mid


## 11. Desempeño mínimo operativo

Como este proyecto se enfoca en MLOps para pricing, el desempeño mínimo no se define con accuracy, F1-score o RMSE. En su lugar, se define mediante criterios de estabilidad y calidad operativa:

- Los percentiles de precio y cantidad deben ser monotónicos.
- La proporción de recomendaciones fuera de una banda válida debe mantenerse baja.
- La proporción de ajustes P20 debe estar controlada.
- La proporción de casos Red y High Priority debe estar controlada.
- Las bandas colapsadas se monitorean, pero no se interpretan como falla crítica por sí solas.

In [15]:
operational_performance = pd.DataFrame([
    {
        "metric": "price_percentiles_monotonic_rate",
        "value": feature_table_scored_final["price_percentiles_monotonic"].mean(),
        "minimum_expected": 0.99,
        "passed": feature_table_scored_final["price_percentiles_monotonic"].mean() >= 0.99
    },
    {
        "metric": "qty_percentiles_monotonic_rate",
        "value": feature_table_scored_final["qty_percentiles_monotonic"].mean(),
        "minimum_expected": 0.99,
        "passed": feature_table_scored_final["qty_percentiles_monotonic"].mean() >= 0.99
    },
    {
        "metric": "recommended_outside_valid_p20_p85_rate",
        "value": feature_table_scored_final["recommended_outside_valid_p20_p85"].mean(),
        "minimum_expected": "<= 0.01",
        "passed": feature_table_scored_final["recommended_outside_valid_p20_p85"].mean() <= 0.01
    },
    {
        "metric": "p20_was_adjusted_rate",
        "value": feature_table_scored_final["P20_Was_Adjusted"].mean(),
        "minimum_expected": "<= 0.05",
        "passed": feature_table_scored_final["P20_Was_Adjusted"].mean() <= 0.05
    },
    {
        "metric": "red_status_rate",
        "value": (feature_table_scored_final["recommendation_status_final"] == "Red").mean(),
        "minimum_expected": "<= 0.01",
        "passed": (feature_table_scored_final["recommendation_status_final"] == "Red").mean() <= 0.01
    },
    {
        "metric": "high_priority_rate",
        "value": (feature_table_scored_final["review_priority_final"] == "High").mean(),
        "minimum_expected": "<= 0.02",
        "passed": (feature_table_scored_final["review_priority_final"] == "High").mean() <= 0.02
    },
    {
        "metric": "collapsed_price_band_rate",
        "value": feature_table_scored_final["price_band_is_collapsed"].mean(),
        "minimum_expected": "monitor_only",
        "passed": True
    }
])

operational_performance

,metric,value,minimum_expected,passed
0,price_percentiles_monotonic_rate,1.000000,0.99,True
1,qty_percentiles_monotonic_rate,1.000000,0.99,True
2,recommended_outside_valid_p20_p85_rate,0.000000,<= 0.01,True
3,p20_was_adjusted_rate,0.003127,<= 0.05,True
4,red_status_rate,0.000000,<= 0.01,True
5,high_priority_rate,0.000000,<= 0.02,True
6,collapsed_price_band_rate,0.576016,monitor_only,True


### Interpretación del desempeño mínimo

El baseline operativo cumple con los criterios mínimos definidos para esta corrida. La métrica de bandas colapsadas se clasifica como `monitor_only`, porque representa baja variabilidad histórica de precio o concentración de ventas en un solo nivel de precio.

Esta métrica deberá compararse contra futuras corridas para identificar si la cobertura del sistema mejora o se deteriora.

## 12. Triggers críticos

Los triggers definen cuándo una señal pasa de observación a alerta crítica.

En esta primera corrida base no se identifican casos Red. Sin embargo, en futuras corridas se activaría una alerta crítica si aparecen recomendaciones fuera de una banda P20-P85 válida, si la proporción de casos High Priority aumenta significativamente, o si las validaciones estructurales de percentiles se deterioran.

In [16]:
trigger_thresholds = pd.DataFrame([
    {
        "metric": "recommended_outside_valid_p20_p85_rate",
        "green": "0%",
        "yellow": ">0% and <=1%",
        "red": ">1%",
        "action": "Revisar recomendaciones fuera de banda válida"
    },
    {
        "metric": "red_status_rate",
        "green": "0%",
        "yellow": ">0% and <=1%",
        "red": ">1%",
        "action": "Escalar revisión del output"
    },
    {
        "metric": "high_priority_rate",
        "green": "<=2%",
        "yellow": ">2% and <=5%",
        "red": ">5%",
        "action": "Priorizar revisión de casos con alto impacto"
    },
    {
        "metric": "p20_was_adjusted_rate",
        "green": "<=5%",
        "yellow": ">5% and <=10%",
        "red": ">10%",
        "action": "Revisar margen mínimo, costos y reglas de ajuste"
    },
    {
        "metric": "price_percentiles_monotonic_rate",
        "green": ">=99%",
        "yellow": "95%-99%",
        "red": "<95%",
        "action": "Revisar cálculo de percentiles de precio"
    },
    {
        "metric": "qty_percentiles_monotonic_rate",
        "green": ">=99%",
        "yellow": "95%-99%",
        "red": "<95%",
        "action": "Revisar cálculo de percentiles de cantidad"
    },
    {
        "metric": "collapsed_price_band_rate",
        "green": "monitor_only",
        "yellow": "incremento vs baseline",
        "red": "incremento fuerte vs baseline",
        "action": "Revisar cobertura y variabilidad histórica"
    }
])

trigger_thresholds

,metric,green,yellow,red,action
0,recommended_outside_valid_p20_p85_rate,0%,>0% and <=1%,>1%,Revisar recomendaciones fuera de banda válida
1,red_status_rate,0%,>0% and <=1%,>1%,Escalar revisión del output
2,high_priority_rate,<=2%,>2% and <=5%,>5%,Priorizar revisión de casos con alto impacto
3,p20_was_adjusted_rate,<=5%,>5% and <=10%,>10%,"Revisar margen mínimo, costos y reglas de ajuste"
4,price_percentiles_monotonic_rate,>=99%,95%-99%,<95%,Revisar cálculo de percentiles de precio
5,qty_percentiles_monotonic_rate,>=99%,95%-99%,<95%,Revisar cálculo de percentiles de cantidad
6,collapsed_price_band_rate,monitor_only,incremento vs baseline,incremento fuerte vs baseline,Revisar cobertura y variabilidad histórica


In [17]:
def classify_global_trigger(row):
    metric = row["metric"]
    value = row["value"]

    if metric == "recommended_outside_valid_p20_p85_rate":
        if value > 0.01:
            return "Red"
        elif value > 0:
            return "Yellow"
        else:
            return "Green"

    if metric == "red_status_rate":
        if value > 0.01:
            return "Red"
        elif value > 0:
            return "Yellow"
        else:
            return "Green"

    if metric == "high_priority_rate":
        if value > 0.05:
            return "Red"
        elif value > 0.02:
            return "Yellow"
        else:
            return "Green"

    if metric == "p20_was_adjusted_rate":
        if value > 0.10:
            return "Red"
        elif value > 0.05:
            return "Yellow"
        else:
            return "Green"

    if metric in ["price_percentiles_monotonic_rate", "qty_percentiles_monotonic_rate"]:
        if value < 0.95:
            return "Red"
        elif value < 0.99:
            return "Yellow"
        else:
            return "Green"

    if metric == "collapsed_price_band_rate":
        return "Monitor Only"

    return "Not Defined"


operational_performance["trigger_status"] = operational_performance.apply(
    classify_global_trigger,
    axis=1
)

operational_performance

,metric,value,minimum_expected,passed,trigger_status
0,price_percentiles_monotonic_rate,1.000000,0.99,True,Green
1,qty_percentiles_monotonic_rate,1.000000,0.99,True,Green
2,recommended_outside_valid_p20_p85_rate,0.000000,<= 0.01,True,Green
3,p20_was_adjusted_rate,0.003127,<= 0.05,True,Green
4,red_status_rate,0.000000,<= 0.01,True,Green
5,high_priority_rate,0.000000,<= 0.02,True,Green
6,collapsed_price_band_rate,0.576016,monitor_only,True,Monitor Only


## 13. Preparación para drift futuro

El baseline actual es una línea de referencia. Las métricas de drift se usarán cuando exista una nueva corrida comparable.

Se dejan preparadas funciones para:

- PSI numérico.
- PSI categórico.
- KS test.

Estas funciones no deben usarse para concluir drift real contra una muestra del mismo dataset, sino para comparar una corrida futura contra esta línea base.

In [18]:
from scipy.stats import ks_2samp

def calculate_psi(expected, actual, buckets=10, eps=1e-6):
    expected = pd.Series(expected).replace([np.inf, -np.inf], np.nan).dropna()
    actual = pd.Series(actual).replace([np.inf, -np.inf], np.nan).dropna()

    if expected.empty or actual.empty:
        return np.nan

    breakpoints = expected.quantile(np.linspace(0, 1, buckets + 1)).values
    breakpoints = np.unique(breakpoints)

    if len(breakpoints) < 3:
        return np.nan

    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    expected_bins = pd.cut(expected, bins=breakpoints, include_lowest=True)
    actual_bins = pd.cut(actual, bins=breakpoints, include_lowest=True)

    expected_dist = expected_bins.value_counts(normalize=True, sort=False).replace(0, eps)
    actual_dist = actual_bins.value_counts(normalize=True, sort=False).replace(0, eps)

    return ((actual_dist - expected_dist) * np.log(actual_dist / expected_dist)).sum()


def calculate_categorical_psi(expected, actual, eps=1e-6):
    expected = pd.Series(expected).astype("object").fillna("Missing")
    actual = pd.Series(actual).astype("object").fillna("Missing")

    categories = sorted(set(expected.unique()).union(set(actual.unique())))

    expected_dist = expected.value_counts(normalize=True).reindex(categories, fill_value=0).replace(0, eps)
    actual_dist = actual.value_counts(normalize=True).reindex(categories, fill_value=0).replace(0, eps)

    return ((actual_dist - expected_dist) * np.log(actual_dist / expected_dist)).sum()


def calculate_ks(expected, actual):
    expected = pd.Series(expected).replace([np.inf, -np.inf], np.nan).dropna()
    actual = pd.Series(actual).replace([np.inf, -np.inf], np.nan).dropna()

    if expected.empty or actual.empty:
        return np.nan, np.nan

    result = ks_2samp(expected, actual)
    return result.statistic, result.pvalue


def classify_psi(psi):
    if pd.isna(psi):
        return "Not Available"
    elif psi < 0.10:
        return "Green"
    elif psi <= 0.25:
        return "Yellow"
    else:
        return "Red"

## 14. Resumen final y exportación de artefactos

Los artefactos exportados permiten versionar el baseline y alimentar futuras etapas de MLOps:

- Resumen del baseline.
- Evaluación de desempeño operativo.
- Feature table con semáforo y score.
- Cola de revisión priorizada.
- Thresholds de triggers.

In [19]:
baseline_summary = pd.DataFrame([
    {
        "baseline_version": BASELINE_VERSION,
        "run_date": RUN_DATE,
        "n_rows": len(feature_table_scored_final),
        "n_unique_kpn": feature_table_scored_final["kpn"].nunique(),
        "n_unique_vparea": feature_table_scored_final["vpareadescription"].nunique(),
        "n_unique_distysegment": feature_table_scored_final["distysegment"].nunique(),
        "green_rate": (feature_table_scored_final["recommendation_status_final"] == "Green").mean(),
        "yellow_rate": (feature_table_scored_final["recommendation_status_final"] == "Yellow").mean(),
        "red_rate": (feature_table_scored_final["recommendation_status_final"] == "Red").mean(),
        "low_priority_rate": (feature_table_scored_final["review_priority_final"] == "Low").mean(),
        "medium_priority_rate": (feature_table_scored_final["review_priority_final"] == "Medium").mean(),
        "high_priority_rate": (feature_table_scored_final["review_priority_final"] == "High").mean(),
        "price_band_collapsed_rate": feature_table_scored_final["price_band_is_collapsed"].mean(),
        "outside_valid_p20_p85_rate": feature_table_scored_final["recommended_outside_valid_p20_p85"].mean(),
        "p20_adjusted_rate": feature_table_scored_final["P20_Was_Adjusted"].mean(),
        "near_extreme_zone_rate": feature_table_scored_final["recommended_near_extreme_zone"].mean()
    }
])

baseline_summary

,baseline_version,run_date,n_rows,n_unique_kpn,n_unique_vparea,n_unique_distysegment,green_rate,yellow_rate,red_rate,low_priority_rate,medium_priority_rate,high_priority_rate,price_band_collapsed_rate,outside_valid_p20_p85_rate,p20_adjusted_rate,near_extreme_zone_rate
0,baseline_mlops_pricing_v1,2026-05-25,8314,6982,2,2,0.61631,0.38369,0.0,0.954294,0.045706,0.0,0.576016,0.0,0.003127,0.381285


In [20]:
OUTPUT_DIR = f"{DIR_PATH}/avance3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

baseline_summary.to_csv(f"{OUTPUT_DIR}/avance3_baseline_summary_equipo46.csv", index=False)
operational_performance.to_csv(f"{OUTPUT_DIR}/avance3_operational_performance_equipo46.csv", index=False)
trigger_thresholds.to_csv(f"{OUTPUT_DIR}/avance3_trigger_thresholds_equipo46.csv", index=False)

feature_table_scored_final.to_csv(
    f"{OUTPUT_DIR}/avance3_feature_table_scored_equipo46.csv",
    index=False
)

review_queue.to_csv(
    f"{OUTPUT_DIR}/avance3_review_queue_equipo46.csv",
    index=False
)

print("Archivos exportados en:", OUTPUT_DIR)

Archivos exportados en: /content/drive/MyDrive/Maestria/Yageo-MLOPS/avance3_outputs


## 15. Conclusiones

En este avance se construyó un baseline MLOps para un sistema de recomendaciones de precios. La línea base no busca reemplazar la lógica económica del motor de pricing, sino monitorear su estabilidad operativa.

El baseline integra métricas de calidad de datos, consistencia de percentiles, estabilidad del output, cobertura de bandas P20-P85, impacto económico y una cola de revisión priorizada.

El análisis mostró que una proporción relevante de registros tiene bandas P20-P85 colapsadas. Estos casos no se consideran errores críticos, sino señales de baja variabilidad histórica o cobertura limitada. Por ello, se separó el outside original de un outside real sobre banda válida.

Con esta corrección, la corrida base no presenta recomendaciones Red ni casos High Priority. Esto sugiere que el sistema se encuentra estable bajo los criterios definidos. El siguiente paso natural es comparar futuras corridas contra este baseline usando PSI, KS, cambios de proporciones y los triggers definidos.